In [16]:
import pandas as pd
import holidays
import numpy as np
import matplotlib.pyplot as plt
import os
import re

In [17]:
test1 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_00.csv")
test2 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_01.csv")
test3 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_02.csv")
test4 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_03.csv")
test5 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_04.csv")
test6 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_05.csv")
test7 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_06.csv")
test8 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_07.csv")
test9 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_08.csv")
test10 = pd.read_csv("D:/새 폴더 (2)/open/test/TEST_09.csv")

In [18]:
train = pd.read_csv("D:/새 폴더 (2)/open/train/train.csv")

외부데이터 - 공휴일 데이터

In [19]:
! pip install holidays


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\tree4\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [20]:
import holidays

In [21]:
kr_holidays = holidays.KR(years=[2023,2024, 2025])
holiday_dict = dict(kr_holidays.items())

holiday_df = pd.DataFrame(
    list(holiday_dict.items()),
    columns=['date', 'holiday_name']
)

print(holiday_df)

          date              holiday_name
0   2024-01-01                      신정연휴
1   2024-02-10                        설날
2   2024-02-09                     설날 전날
3   2024-02-11                    설날 다음날
4   2024-03-01                       삼일절
5   2024-05-15                    부처님오신날
6   2024-05-05                      어린이날
7   2024-06-06                       현충일
8   2024-08-15                       광복절
9   2024-10-03                       개천절
10  2024-10-09                       한글날
11  2024-09-17                        추석
12  2024-09-16                     추석 전날
13  2024-09-18                    추석 다음날
14  2024-12-25                     기독탄신일
15  2024-04-10                  국회의원 선거일
16  2024-02-12                  설날 대체 휴일
17  2024-05-06                어린이날 대체 휴일
18  2024-10-01                     국군의 날
19  2025-01-01                      신정연휴
20  2025-01-29                        설날
21  2025-01-28                     설날 전날
22  2025-01-30                    설날 다음날
23  2025-03-01  

In [22]:
holiday_df.to_csv('holiday.csv')

데이터 전처리

In [23]:
test_files = [f'D:/새 폴더 (2)/open/test/TEST_{i:02d}.csv' for i in range(10)]
test_df = [pd.read_csv(fp, parse_dates = ['영업일자']) for fp in test_files]
test = pd.concat(test_df, ignore_index=True)

test = test.rename(columns={
    '영업일자' : 'date',
    '영업장명_메뉴명' : 'store_menu',
    '매출수량' : 'sales_count'
})

In [24]:
train = pd.read_csv("D:/새 폴더 (2)/open/train/train.csv", parse_dates = ['영업일자'])
train = train.rename(columns={
    '영업일자' : 'date',
    '영업장명_메뉴명' : 'store_menu',
    '매출수량' : 'sales_count'
})

In [25]:
def add_domain_features(df, holiday_df, date_col='date'):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col]).dt.normalize()

    # holiday_df 전처리
    hd = holiday_df.copy()
    if 'date' not in hd.columns:
        for c in hd.columns:
            if ('date' in c.lower()) or ('일자' in c):
                hd = hd.rename(columns={c: 'date'})
                break
        else:
            raise ValueError("holiday_df에 날짜 컬럼이 없습니다. 'date' 또는 '*일자' 필요")
    hd['date'] = pd.to_datetime(hd['date']).dt.normalize()

    # 공휴일 여부 플래그 생성
    # holiday_df에서 날짜만 추출 후 중복 제거 -> is_holiday=1 부여
    hol = hd[['date']].drop_duplicates().assign(is_holiday=1)
    # df와 merge하여 해당 날짜가 공휴일이면 1, 아니면 0
    df = df.merge(hol, on='date', how='left')
    df['is_holiday'] = df['is_holiday'].fillna(0).astype('int8')

    # 날짜 기반 파생 변수 생성
    w = df[date_col].dt.weekday # 요일 (월=0 ... 일 =6)
    df['weekday']    = w.astype('int8') # 요일 숫자 저장
    df['is_weekend'] = w.isin([5,6]).astype('int8') # 주말 여부(토 =5, 일=6)
    df['month']      = df[date_col].dt.month.astype('int8') # 월
    df['quarter']    = df[date_col].dt.quarter.astype('int8') # 분기(1-4)
    df['season']     = ((df['month'] % 12)//3 + 1).astype('int8') # 계절(1: 겨울, 2:봄, 3:여름, 4:가을)
    df['day']        = df[date_col].dt.day.astype('int8') # 일 (1-31)

    # ---- 휴일까지 거리 계산 ----
    # holiday_df 날짜 배열을 오름차순으로 정렬 (datetime64[D])
    hol_days = np.sort(hd['date'].to_numpy(dtype='datetime64[D]'))  # 정렬된 공휴일 배열
    # df의 날짜 배열(datetime64[D] 타입)
    d_days   = df[date_col].to_numpy(dtype='datetime64[D]')

    # 각 날짜에 대해 다음 공휴일 index (왼쪽 삽입 위치)
    idx_next = np.searchsorted(hol_days, d_days, side='left')
    # 각 날짜의 이전 공휴일 index (오른쪽 삽입 위치 -1)
    idx_prev = np.searchsorted(hol_days, d_days, side='right') - 1

    # 기본값: NA 배열 생성
    days_to_next      = np.full(len(df), np.nan)
    days_since_last   = np.full(len(df), np.nan)

    # 다음 공휴일까지 남은 일수
    m_next = idx_next < len(hol_days) # 다음 공휴일이 존재하는 날짜만
    days_to_next[m_next] = (
        hol_days[idx_next[m_next]] - d_days[m_next]
    ).astype('timedelta64[D]').astype(int)

    # 지난 공휴일로부터 경과 일수
    m_prev = idx_prev >= 0 # 이전 공휴일이 존재하는 날짜만
    days_since_last[m_prev] = (
        d_days[m_prev] - hol_days[idx_prev[m_prev]]
    ).astype('timedelta64[D]').astype(int)

    # pandas nullable 정수형(Int 16)으로 변호나 -> 결측 허용
    df['days_to_next_holiday']    = pd.Series(days_to_next).astype('Int16')
    df['days_since_last_holiday'] = pd.Series(days_since_last).astype('Int16')

    # 휴일 당일은 두 값 모두 0으로 설정
    m_hol = df['is_holiday'] == 1
    df.loc[m_hol, ['days_to_next_holiday', 'days_since_last_holiday']] = 0

    return df


In [26]:
train_enriched = add_domain_features(train,holiday_df)
test_enriched = add_domain_features(test, holiday_df)

display(train_enriched.head())
display(test_enriched.head())

,date,store_menu,sales_count,is_holiday,weekday,is_weekend,month,quarter,season,day,days_to_next_holiday,days_since_last_holiday
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0,1,6,1,1,1,1,1,0,0
1,2023-01-02,느티나무 셀프BBQ_1인 수저세트,0,0,0,0,1,1,1,2,19,1
2,2023-01-03,느티나무 셀프BBQ_1인 수저세트,0,0,1,0,1,1,1,3,18,2
3,2023-01-04,느티나무 셀프BBQ_1인 수저세트,0,0,2,0,1,1,1,4,17,3
4,2023-01-05,느티나무 셀프BBQ_1인 수저세트,0,0,3,0,1,1,1,5,16,4


,date,store_menu,sales_count,is_holiday,weekday,is_weekend,month,quarter,season,day,days_to_next_holiday,days_since_last_holiday
0,2024-06-16,느티나무 셀프BBQ_1인 수저세트,2,0,6,1,6,2,3,16,60,10
1,2024-06-17,느티나무 셀프BBQ_1인 수저세트,0,0,0,0,6,2,3,17,59,11
2,2024-06-18,느티나무 셀프BBQ_1인 수저세트,0,0,1,0,6,2,3,18,58,12
3,2024-06-19,느티나무 셀프BBQ_1인 수저세트,0,0,2,0,6,2,3,19,57,13
4,2024-06-20,느티나무 셀프BBQ_1인 수저세트,4,0,3,0,6,2,3,20,56,14


상관관계 분석

In [27]:
# 숫자형 피처 목록 정의
# (공휴일, 요일, 주말 여부, 월, 분기, 계절, 일, 다음 공휴일까지 일수, 지난 공휴일부터 일수)
num_feats = [
    'is_holiday','weekday','is_weekend','month','quarter','season','day','days_to_next_holiday','days_since_last_holiday'
]

# 결측 채움(거리 피처에 NA -> 큰 수로 처리(999))
corr_df_base = train_enriched.copy()
corr_df_base['days_to_next_holiday'] = corr_df_base['days_to_next_holiday'].fillna(999)
corr_df_base['days_since_lst_holiday'] = corr_df_base['days_since_last_holiday'].fillna(999)

# 전체 pearson(선형 관계) 
# numeric_only = True -> 숫자형 열만 사용
# 'sales_count'와 각 피처 간 상관계수 추출
pearson = corr_df_base[num_feats + ['sales_count']].corr(numeric_only=True)['sales_count'].sort_values(ascending=False)

# spearman 순위(서열) 기반 상관관계
# drop('sales_count') -> 자기 자신과의 상관계수(1.0) 제거
spearman = corr_df_base[num_feats + ['sales_count']].corr(method='spearman', numeric_only=True)['sales_count'].drop('sales_count').sort_values(ascending=False)

print('=== Pearson correlation (overall) ===')
print(pearson.sort_values(ascending=False))
print('\n=== Spearman correlation (overall) ===')
print(spearman.sort_values(ascending=False))

=== Pearson correlation (overall) ===
sales_count                1.000000
weekday                    0.055531
is_weekend                 0.050485
is_holiday                 0.033458
day                        0.003585
quarter                   -0.021363
days_to_next_holiday      -0.030738
days_since_last_holiday   -0.031727
month                     -0.037035
season                    -0.063823
Name: sales_count, dtype: float64

=== Spearman correlation (overall) ===
weekday                    0.127872
is_weekend                 0.093938
quarter                    0.079606
is_holiday                 0.049070
month                      0.045060
day                        0.018746
season                    -0.005791
days_since_last_holiday   -0.011947
days_to_next_holiday      -0.077629
Name: sales_count, dtype: float64


결론.

-weekday/ is_weekend/ is_holiday가 양수값: 요일,주말, 공휴일이 매출 변화방향을 갖긴 함.
다만, 값이 0.03~0.13 수준으로 작음. 전체 데이터를 한데 섞어 보면 선형 상관은 약함.

-Spearman > Pearson: 선형은 약하지만 '순서 경향'은 존재.
트리/부스팅처럼 비선형 모델이 유리, 단순 선형회귀는 한계.

-season, month, quarter가 혼조: 시즌성은 분명 있지만 업장마다 패턴이 달라 전체 평균에서 희석됨.

-days_to_next_holiday가 음수: 다음 휴일까지 멀어질수록 매출이 약해지는 경향이 일어날 수도 있다는 약한 신호(미약)

요약. 전체 상관은 약함. 업장/메뉴별로 패턴이 달라져 섞이면 희석되니, 세분(업장,메뉴단위) 모델링이나 상호작용이 중요.

In [30]:
# 업장별 상관
# store_menu에서 '_' 앞부분만 추출해 업장명(store)생성
# 예) '느티나무 셀프BBQ_1인 수저세트' -> '느티나무 셀프 BBQ'
# '_'가 없는 경우는 그대로 사용
corr_df_base['store'] = corr_df_base['store_menu'].str.split('_').str[0]

rows = []

# === 업장별로 그룹을 나누어 반복===
for k,g in corr_df_base.groupby('store'): # k=업장명, g=그 업장 데이터
    # 분석에 필요한 피처와 매출 컬럼만 복사
    gg = g[num_feats + ['sales_count']].copy()
    # 공휴일 거리 피처의 결측을 999로 채움
    gg['days_to_next_holiday'] = gg['days_to_next_holiday'].fillna(999)
    gg['days_since_last_holiday'] = gg['days_since_last_holiday'].fillna(999)

    # 데이터가 너무 적으면(30행 미만) 분석 제외
    if len(gg) < 30:
        continue

    # pearson 상관계수 계산(sales_count 기준)
    # numeric_only=True -> 숫자형 데이터만 계산
    p = gg.corr(numeric_only=True)['sales_count'].drop('sales_count')
    # 상관계수 결과를 DataFrame으로 변환하여 rows에 추가
    rows.append(
        pd.DataFrame({
            'store': [k]*len(p), # 해당 업장명 반복
            'feature':p.index, # 피처 이름
            'pearson':p.values # pearson 상관계수 값
        })
    )

# 업장별 상관계수 전체 테이블 결합
store_corr = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['store','feature','pearson'])

# 업장별 pearson 상관계수 상위 3개 피처 추출
store_top3 = (store_corr.sort_values(['store','pearson'], ascending=[True,False]).groupby('store').head(3))
print('=== Per-store top3 Pearson features ===')
print(store_top3)

=== Per-store top3 Pearson features ===
         store                  feature   pearson
5   느티나무 셀프BBQ                   season  0.109380
3   느티나무 셀프BBQ                    month  0.081600
7   느티나무 셀프BBQ     days_to_next_holiday  0.081529
10          담하                  weekday  0.087637
11          담하               is_weekend  0.074552
13          담하                  quarter  0.058434
19        라그로타                  weekday  0.176732
20        라그로타               is_weekend  0.140036
18        라그로타               is_holiday  0.069569
31        미라시아                  quarter  0.074323
30        미라시아                    month  0.064393
32        미라시아                   season  0.044579
40         연회장                  quarter  0.011989
44         연회장  days_since_last_holiday  0.009523
39         연회장                    month  0.009317
45       카페테리아               is_holiday  0.059616
46       카페테리아                  weekday  0.052285
47       카페테리아               is_weekend  0.050330
56       포

결론. 

-느티나무 셀프 BBQ: season, month, days_to_next_holiday -> 계절,연휴 영향 큼
-담하: weekday, is_weekend -> 요일/주말 차이 뚜렷
-라그로타: weekday, is_weekend, is_holiday -> 요일, 휴일 민감
-미라시아: quarter. month -> 분기,월 단위 시즌성
-프레스트릿: is_weekend, weekday, is_holiday -> 주말/휴일형 수요
-화담숲주막/카페: season,quarter,month 값이 0.20~0.32로 높음 -> 강력한 계절성

요약. 업장별로 잘먹히는 피처가 다름.
담하,미라시아가 평가 가중치가 높은데, 각각 요일/주말, 분기/월 축의 피처가 상대적으로 유효

A안 모델링 흐름 - 전역 모델 기반, 상관계수 결과 참고

28 -> 7일 윈도 생성 (시퀀스/트리 공용)

In [32]:
def make_windows_fast(df, group_col='store_menu', date_col='date',
                      target_col='sales_count', feature_cols=None,
                      lookback=28, horizon=7, fill_distance=999):

    """ 목적:
        - (그룹별) 시계열 데이터를 슬라이딩 윈도우로 잘라
        입력 x: 과거 lookback 일 x 피처
        출력 y: 미래 horizon일의 타깃
        - 벡터화된 인덱싱으로 빠르게 생성
        
        반환:
        X_seq: (총윈도우수, lookback, F) float32
        y_seq: (총윈도우수, horizon) float32
        MEAT: 각 윈도우의 메타정보(그룹, 기준일, 예측구간 시작/끝)
        feature_cols: 실제 사용된 피처 목록
    """


    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # 거리형 피처 결측 채우기(모델 입력에서 NA 방지)
    for c in ['days_to_next_holiday','days_since_last_holiday']:
        if c in df.columns: df[c] = df[c].fillna(fill_distance)

    # 사용할 피처 목록 자동 구성(미지정 시)
    if feature_cols is None:
        base = ['is_holiday','weekday','is_weekend','month','quarter','season','day',
                'days_to_next_holiday','days_since_last_holiday']
        # 타깃을 과거 정보로도 쓰고 싶으면 'sales_count' 포함
        # base 중 실제 df에 존재하는 칼럼만 사용
        feature_cols = ['sales_count'] + [c for c in base if c in df.columns]

    X_parts, Y_parts, META = [], [], []

    # 그룹(업장/메뉴 등)별로 독립적인 시계열 윈도우 생성
    for k, g in df.groupby(group_col, sort=False):
        # 날짜 순으로 정렬
        g = g.sort_values(date_col).reset_index(drop=True)

        # 필요한 칼럼만 남기고 순서 보장
        cols_need = [date_col] + feature_cols + [target_col]
        cols_need = list(dict.fromkeys(cols_need))  # 중복 제거
        if any(c not in g.columns for c in cols_need): 
            # 필요한 컬럼이 부족한 그룹은 스킵
            continue
        g = g[cols_need]

        # 만들 수 있는 윈도우 개수 n = T - lookback - horizon + 1
        T = len(g); n = T - lookback - horizon + 1
        if n <= 0: continue # 길이가 짧아 윈도우를 못 만드는 그룹은 스킵

        # 넘파이 배열로 변환
        arr = g[feature_cols].to_numpy(np.float32)     # (T,F)
        tgt = g[target_col].to_numpy(np.float32)       # (T,)
        
        # 시작 인덱스 (0 - n-1)를 벡터로
        starts = np.arange(n, dtype=np.int32)

        # 입력 윈도우의 인덱스 모음:
        # 각 시작 s에 대해 [s, s+1, ... , s+ lookback-1]
        in_idx  = starts[:, None] + np.arange(lookback)
        # 출력 타깃 윈도우의 인덱스 모음:
        # 각 시작 s에 대해 [s, s+1, ... , s+ horizon-1]
        out_idx = starts[:, None] + lookback + np.arange(horizon)

        # 고급 인덱싱으로 한 번에 자르기(복수 윈도우 병렬 추출)
        # 결과: X: (n, lookback, F), y: (n, horizon)
        X_parts.append(arr[in_idx])            # (n,28,F)
        Y_parts.append(tgt[out_idx])           # (n,7)
        
        # 각 윈도우의 메타 정보 저장(검증/제출 시 매핑용)
        META.extend({
            'group': k,
            # anchor_date: 입력 구간의 마지막 날짜(=예측일 기준)
            'anchor_date': g.loc[s+lookback-1, date_col],
            'horizon_start': g.loc[s+lookback, date_col],
            'horizon_end': g.loc[s+lookback+horizon-1, date_col],
        } for s in starts)

    # 그룹별 조각을 하나로 연결(없으면 빈 배열)
    X_seq = np.concatenate(X_parts, 0) if X_parts else np.empty((0, lookback, len(feature_cols)), np.float32)
    y_seq = np.concatenate(Y_parts, 0) if Y_parts else np.empty((0, horizon), np.float32)
    return X_seq, y_seq, META, feature_cols

def flatten_for_tree(X_3d):
    """목적:
    - (n,L,F) 형태의 시계열 입력을 트리/선형 모델용 2D 평탄화
    예: LightGBM/ XGBoost / LinearModel 등에 바로 투입
    
    입력:
    - X_3d: (n, lookback, num_features)
    반환:
     (n, lookback*num_features)"""
    n, L, F = X_3d.shape
    return X_3d.reshape(n, L*F)


In [30]:
train_enriched = add_domain_features(train, holiday_df, date_col='date')
X_seq, y_seq, meta, used_feats = make_windows_fast(train_enriched)
X_flat = flatten_for_tree(X_seq)
print(X_seq.shape, y_seq.shape, X_flat.shape, used_feats[:5], meta[0] if meta else None)

(96114, 28, 10) (96114, 7) (96114, 280) ['sales_count', 'is_holiday', 'weekday', 'is_weekend', 'month'] {'group': '느티나무 셀프BBQ_1인 수저세트', 'anchor_date': Timestamp('2023-01-28 00:00:00'), 'horizon_start': Timestamp('2023-01-29 00:00:00'), 'horizon_end': Timestamp('2023-02-04 00:00:00')}


In [31]:
def make_window_level_features(X_seq, used_feats):
    N, L, F = X_seq.shape
    idx_sc = used_feats.index('sales_count')
    sc = X_seq[:, :, idx_sc].astype(float)  # (N,L)

    def lag_k(k): return sc[:, -k] if L >= k else np.zeros(N)
    def tail_mean(k): k=min(k,L); return sc[:, -k:].mean(1)
    def tail_std(k):  k=min(k,L); return sc[:, -k:].std(1)

    # lags
    lag1, lag3, lag7, lag14, lag28 = lag_k(1), lag_k(3), lag_k(7), lag_k(14), lag_k(28)
    # rolling
    r7_mean, r7_std = tail_mean(7), tail_std(7)
    r14_mean, r14_std = tail_mean(14), tail_std(14)
    r28_mean, r28_std = tail_mean(28), tail_std(28)
    # 추세
    t = np.arange(L, dtype=float)
    t = (t - t.mean()) / (t.std() + 1e-8)
    slope = ((sc * t).mean(1) - sc.mean(1) * t.mean()) / (t.var() + 1e-8)

    feats = [lag1, lag3, lag7, lag14, lag28,
             r7_mean, r14_mean, r28_mean, r7_std, r14_std, r28_std, slope]

    # 같은 요일 평균 / 주말·주중 평균 (있으면)
    if 'weekday' in used_feats:
        wd = X_seq[:, :, used_feats.index('weekday')]
        anchor = wd[:, -1][:, None]
        m = (wd == anchor)
        same_wd_mean = (sc * m).sum(1) / np.maximum(m.sum(1), 1)
        feats.append(same_wd_mean)

    if 'is_weekend' in used_feats:
        we = X_seq[:, :, used_feats.index('is_weekend')]
    elif 'weekday' in used_feats:
        wd = X_seq[:, :, used_feats.index('weekday')]
        we = (wd >= 5).astype(float)
    else:
        we = np.zeros_like(sc)

    weekend_mean = (sc * we).sum(1) / np.maximum(we.sum(1), 1)
    weekday_mean = (sc * (1 - we)).sum(1) / np.maximum((1 - we).sum(1), 1)
    feats += [weekend_mean, weekday_mean]

    return np.column_stack(feats)

# 트리 입력 확장
X_win = make_window_level_features(X_seq, used_feats)   # (N,K)
X_aug = np.hstack([X_flat, X_win])                      # 최종 입력


In [32]:
import numpy as np, lightgbm as lgb

def smape_ignore_zero(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    m = y_true != 0
    if m.sum()==0: return np.nan
    num = np.abs(y_pred[m]-y_true[m]); den = (np.abs(y_true[m])+np.abs(y_pred[m]))/2
    return np.mean(num/np.maximum(den,1e-8))*100

def iter_time_folds(order, n_splits=5):
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for tr, va in tscv.split(order): yield order[tr], order[va]

# anchor_date 정렬 인덱스
anchor = np.array([m['anchor_date'] for m in meta])
order = np.argsort(anchor)

# 업장 가중치(일단 OFF)
def make_sample_weights(meta, upweight=1.0):
    w = np.ones(len(meta), float)
    if upweight!=1.0:
        for i,m in enumerate(meta):
            if str(m['group']).split('_')[0] in ('담하','미라시아'):
                w[i] = upweight
    return w

sample_weight = make_sample_weights(meta, upweight=1.0)

def feval_smape_plain(y_true, y_pred):
    return ("smape", smape_ignore_zero(y_true, np.clip(y_pred,0,None)), False)

def make_feval_smape_log(shift):
    def _f(y_true, y_pred):
        y_hat = np.expm1(y_pred) - shift
        y_hat = np.clip(y_hat, 0, None)
        return ("smape", smape_ignore_zero(y_true, y_hat), False)
    return _f

use_log = True; es_rounds = 100
H = y_seq.shape[1]
models = []; cv_scores = []

for h in range(H):
    y = y_seq[:, h].astype(float)
    shift = max(1e-6, -float(np.min(y)) + 1e-6) if use_log else 0.0

    fold_scores, fold_models = [], []
    for tr_idx, va_idx in iter_time_folds(order):
        X_tr, X_va = X_aug[tr_idx], X_aug[va_idx]
        y_tr_raw, y_va_raw = y[tr_idx], y[va_idx]
        w_tr = sample_weight[tr_idx]

        if use_log:
            y_tr = np.log1p(y_tr_raw + shift)
            eval_label = y_va_raw
            feval = make_feval_smape_log(shift)
        else:
            y_tr = y_tr_raw
            eval_label = y_va_raw
            feval = feval_smape_plain

        reg = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=1200, learning_rate=0.05,
            num_leaves=127, max_depth=-1, min_child_samples=20,
            subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.1, reg_lambda=0.1,
            random_state=42, n_jobs=-1
        )
        reg.fit(X_tr, y_tr, sample_weight=w_tr,
                eval_set=[(X_va, eval_label)], eval_metric=feval,
                callbacks=[lgb.early_stopping(stopping_rounds=es_rounds, verbose=False)])

        y_hat = reg.predict(X_va)
        if use_log: y_hat = np.expm1(y_hat) - shift
        y_hat = np.clip(y_hat, 0, None)

        sc = smape_ignore_zero(y_va_raw, y_hat)
        if np.isfinite(sc):
            fold_scores.append(sc); fold_models.append(reg)

    if len(fold_scores)==0:
        models.append(None); print(f"H+{h+1} CV SMAPE: N/A")
    else:
        mean_sc = float(np.mean(fold_scores))
        best_i = int(np.argmin(fold_scores))
        best_md = fold_models[best_i]
        models.append((best_md, shift, use_log))
        best_iter = getattr(best_md, "best_iteration_", None)
        print(f"H+{h+1} CV SMAPE (mean): {mean_sc:.3f}% | Best: {fold_scores[best_i]:.3f}%"
              + (f" (best_iter={best_iter})" if best_iter is not None else ""))
        cv_scores.append(mean_sc)

overall = float(np.mean(cv_scores)) if cv_scores else np.nan
print("\n=== Overall CV SMAPE ===")
print(f"{overall:.3f}%" if np.isfinite(overall) else "N/A")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015838 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15362
[LightGBM] [Info] Number of data points in the train set: 16019, number of used features: 287
[LightGBM] [Info] Start training from score 4.469986


C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033106 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16365
[LightGBM] [Info] Number of data points in the train set: 32038, number of used features: 295
[LightGBM] [Info] Start training from score 4.465058


C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.070851 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16627
[LightGBM] [Info] Number of data points in the train set: 48057, number of used features: 295
[LightGBM] [Info] Start training from score 4.462167


...

C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.190648 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16961
[LightGBM] [Info] Number of data points in the train set: 64076, number of used features: 295
[LightGBM] [Info] Start training from score 4.471150


C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.072756 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 17125
[LightGBM] [Info] Number of data points in the train set: 80095, number of used features: 295
[LightGBM] [Info] Start training from score 4.476806
H+7 CV SMAPE (mean): 62.598% | Best: 54.256% (best_iter=49)

=== Overall CV SMAPE ===
59.925%


C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [33]:
# 최종 재학습
final_models = []
for h in range(H):
    cv_model, shift, use_log = models[h]
    y = y_seq[:, h].astype(float)
    best_iter = getattr(cv_model, "best_iteration_", None) or cv_model.get_params().get("n_estimators", 800)
    params = cv_model.get_params(); params.update(dict(n_estimators=best_iter, objective="regression", n_jobs=-1))
    mdl = lgb.LGBMRegressor(**params)
    y_train = np.log1p(y + shift) if use_log else y
    mdl.fit(X_aug, y_train, sample_weight=sample_weight)
    final_models.append((mdl, shift, use_log))

def build_test_sequences(df_test, holiday_df, used_feats, group_col="store_menu", date_col="date"):
    df = add_domain_features(df_test, holiday_df, date_col)
    df = df.sort_values([group_col, date_col]).reset_index(drop=True)
    X_list, META = [], []
    for g, sub in df.groupby(group_col, sort=False):
        sub = sub.sort_values(date_col).reset_index(drop=True)
        if len(sub)!=28: continue
        if any(c not in sub.columns for c in used_feats): continue
        X_list.append(sub[used_feats].to_numpy(np.float32))
        anchor = pd.to_datetime(sub[date_col].iloc[-1])
        META.append({'group': g, 'anchor_date': anchor,
                     'pred_dates': [anchor + pd.Timedelta(days=i) for i in range(1,8)]})
    if not X_list:
        return np.empty((0,28,len(used_feats)),np.float32), None
    X_seq_t = np.stack(X_list, 0)
    X_flat_t = X_seq_t.reshape(X_seq_t.shape[0], -1)
    X_win_t  = make_window_level_features(X_seq_t, used_feats)
    X_aug_t  = np.hstack([X_flat_t, X_win_t])
    return (X_aug_t, META)

def predict_7(models_with_meta, X_input):
    preds = []
    for mdl, shift, use_log in models_with_meta:
        p = mdl.predict(X_input)
        if use_log: p = np.expm1(p) - shift
        preds.append(np.clip(p, 0, None))
    return np.column_stack(preds)

# 예: 테스트 하나 읽어서 제출 포맷 생성 (대회 포맷에 맞게 수정 필요)
# df_test = pd.read_csv(".../TEST_00.csv")
# X_aug_test, META = build_test_sequences(df_test, holiday_df, used_feats)
# Y_hat = predict_7(final_models, X_aug_test)  # (N,7)
# -> sample_submission 형태에 맞게 매핑 후 저장


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.087854 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17125
[LightGBM] [Info] Number of data points in the train set: 96114, number of used features: 295
[LightGBM] [Info] Start training from score 4.476681
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.090706 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17125
[LightGBM] [Info] Number of data points in the train set: 96114, number of used features: 295
[LightGBM] [Info] Start training from score 4.476382
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.128918 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17125
[LightGBM] [Info] Number of data points in the train set: 96114, number of used features: 295
[LightGBM] [Info] Sta

In [57]:
def _detect_date_col(df):
    for c in ['date','영업일자','일자','dt']:
        if c in df.columns: return c
    # fallback
    for c in df.columns:
        lc = str(c).lower()
        if ('date' in lc) or ('일자' in c) or ('영업일' in c):
            return c
    raise KeyError("날짜 컬럼 못 찾음")

def _detect_group_col(df):
    if 'store_menu' in df.columns: return 'store_menu'
    if '영업장명_메뉴명' in df.columns: return '영업장명_메뉴명'
    # fallback
    for c in df.columns:
        if '메뉴' in c and '영업장' in c:  # ex) 영업장명_메뉴명
            return c
    raise KeyError("그룹 컬럼 못 찾음")

def _detect_sales_col(df):
    # 네 테스트 기준 우선 매핑
    if '매출수량' in df.columns: return '매출수량'
    for a in ['sales_count','sales','qty','quantity','count','판매수량','판매수','수량','매출수량','매출건수','건수']:
        if a in df.columns: return a
    # fallback: 숫자형 중 분산 큰 컬럼
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if num_cols:
        return pd.Series({c: df[c].var() for c in num_cols}).idxmax()
    raise KeyError("판매량 컬럼 못 찾음")


In [58]:
def build_test_sequences_flex(df_test, holiday_df, used_feats):
    date_col  = _detect_date_col(df_test)              # ==> '영업일자'
    group_col = _detect_group_col(df_test)             # ==> '영업장명_메뉴명'

    df = add_domain_features(df_test, holiday_df, date_col=date_col)
    df = df.sort_values([group_col, date_col]).reset_index(drop=True)

    X_list, META = [], []
    for g, sub in df.groupby(group_col, sort=False):
        sub = sub.sort_values(date_col).reset_index(drop=True)
        if len(sub) != 28:    # 룰: 입력 28일
            continue

        # --- sales_count 매핑 ---
        sales_col = _detect_sales_col(sub) if 'sales_count' not in sub.columns else 'sales_count'

        # 다른 피처는 그대로 존재해야 함
        missing_others = [c for c in used_feats if c != 'sales_count' and c not in sub.columns]
        if missing_others:
            print(f"[WARN] {g}: missing {missing_others} → skip"); continue

        # used_feats 순서로 컬럼 구성(sales_count만 대체)
        cols = [sales_col if c=='sales_count' else c for c in used_feats]

        X_28F = sub[cols].to_numpy(np.float32)
        X_flat = X_28F.reshape(1, -1)
        X_win  = make_window_level_features(X_28F[None, ...], used_feats)
        X_aug  = np.hstack([X_flat, X_win])

        anchor = pd.to_datetime(sub[date_col].iloc[-1])
        META.append({'group': g, 'anchor_date': anchor})
        X_list.append(X_aug)

    if not X_list:
        return np.empty((0,0), np.float32), []
    return np.vstack(X_list), META


In [59]:
import os, glob, re
import numpy as np
import pandas as pd

# ===== 경로 설정 =====
SAMPLE_PATH = "D:/새 폴더 (2)/open/sample_submission.csv"
TEST_DIR    = "D:/새 폴더 (2)/open/test"
TEST_GLOB   = os.path.join(TEST_DIR, "TEST_*.csv")
OUT_PATH    = "submission.csv"

# ===== 컬럼 탐지 유틸 =====
def _detect_date_col(df):
    for c in ['date','영업일자','일자','dt']:
        if c in df.columns: return c
    for c in df.columns:
        lc = str(c).lower()
        if ('date' in lc) or ('일자' in c) or ('영업일' in c):
            return c
    raise KeyError("날짜 컬럼 못 찾음")

def _detect_group_col(df):
    if 'store_menu' in df.columns: return 'store_menu'
    if '영업장명_메뉴명' in df.columns: return '영업장명_메뉴명'
    for c in df.columns:
        if '메뉴' in c and '영업장' in c:
            return c
    raise KeyError("그룹 컬럼 못 찾음")

def _detect_sales_col(df):
    if '매출수량' in df.columns: return '매출수량'
    for a in ['sales_count','sales','qty','quantity','count','판매수량','판매수','수량','매출수량','매출건수','건수']:
        if a in df.columns: return a
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if num_cols:
        return pd.Series({c: df[c].var() for c in num_cols}).idxmax()
    raise KeyError("판매량 컬럼 못 찾음")

# ===== 테스트 시퀀스 빌더 =====
def build_test_sequences_flex(df_test, holiday_df, used_feats):
    date_col  = _detect_date_col(df_test)
    group_col = _detect_group_col(df_test)

    df = add_domain_features(df_test, holiday_df, date_col=date_col)
    df = df.sort_values([group_col, date_col]).reset_index(drop=True)

    X_list, META = [], []
    for g, sub in df.groupby(group_col, sort=False):
        sub = sub.sort_values(date_col).reset_index(drop=True)
        if len(sub) != 28:
            continue

        # sales_count 매핑
        sales_col = _detect_sales_col(sub) if 'sales_count' not in sub.columns else 'sales_count'

        # 다른 피처 체크
        missing_others = [c for c in used_feats if c != 'sales_count' and c not in sub.columns]
        if missing_others:
            print(f"[WARN] {g}: missing {missing_others} → skip")
            continue

        cols = [sales_col if c == 'sales_count' else c for c in used_feats]

        X_28F = sub[cols].to_numpy(np.float32)
        X_flat = X_28F.reshape(1, -1)
        X_win  = make_window_level_features(X_28F[None, ...], used_feats)
        X_aug  = np.hstack([X_flat, X_win])

        anchor = pd.to_datetime(sub[date_col].iloc[-1])
        META.append({"group": g, "anchor_date": anchor})
        X_list.append(X_aug)

    if not X_list:
        return np.empty((0,0), np.float32), []
    return np.vstack(X_list), META

# ===== 예측 함수 =====
def predict_7(models_with_meta, X_input):
    preds = []
    for mdl, shift, use_log in models_with_meta:
        p = mdl.predict(X_input)
        if use_log:
            p = np.expm1(p) - shift
        preds.append(np.clip(p, 0, None))
    return np.column_stack(preds)

# ===== 제출 생성 =====
sample = pd.read_csv(SAMPLE_PATH)
submit = sample.copy()
submit.iloc[:, 1:] = 0.0

test_files = sorted(glob.glob(TEST_GLOB))
if not test_files:
    raise FileNotFoundError(f"테스트 파일이 없습니다: {TEST_GLOB}")

filled_cells = 0
missing_keys = 0

for fpath in test_files:
    fname = os.path.basename(fpath)
    test_id = os.path.splitext(fname)[0]

    df_test = pd.read_csv(fpath)
    X_aug_t, META = build_test_sequences_flex(df_test, holiday_df, used_feats)

    if len(META) == 0 or X_aug_t.shape[0] == 0:
        print(f"[WARN] {fname}: usable groups=0 → skip")
        continue

    Y_hat = predict_7(models, X_aug_t)

    for i, m in enumerate(META):
        col = m["group"]
        if col not in submit.columns:
            print(f"[WARN] {col} 컬럼이 sample_submission에 없습니다 → skip")
            continue

        for d, val in enumerate(Y_hat[i], start=1):
            row_key = f"{test_id}+{d}일"
            ridx = submit.index[submit['영업일자'] == row_key]
            if len(ridx) >= 1:
                submit.loc[ridx[0], col] = float(val)
                filled_cells += 1
            else:
                missing_keys += 1

print(f"채운 셀: {filled_cells} | 누락 row_keys: {missing_keys}")
submit.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved: {OUT_PATH}")


C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\tree4\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packag

채운 셀: 13510 | 누락 row_keys: 0
Saved: submission.csv


B안 모델링 매장별 상관관계 기반 피처 선택 + 개별 모델